# DealX AI Engine - Exploration Notebook

Scratch notebook for testing the forecasting model, Deal Integrity Score, spec matcher, and negotiation bot on the bundled synthetic sample data (`../sample_data/`) before the real scraper/DB is producing data.

Run cells top to bottom.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.join('..', 'forecasting'))
sys.path.insert(0, os.path.join('..', 'deal_integrity'))
sys.path.insert(0, os.path.join('..', 'llm_matching'))
sys.path.insert(0, os.path.join('..', 'negotiation_bot'))

SAMPLE_DIR = os.path.join('..', 'sample_data')
with open(os.path.join(SAMPLE_DIR, 'price_history_sample.json')) as f:
    price_data = json.load(f)

list(price_data.keys())

## 1. Forecasting

Try each sample product. `prod_003` has short history on purpose, to exercise the scikit-learn fallback path.

In [ ]:
from predict import predict_forecast

product_id = 'prod_001'
result = predict_forecast(product_id, price_data[product_id]['history'], days=30)
print('model used:', result['model_type'])
result['forecast'][:5]

In [ ]:
import matplotlib.pyplot as plt

history = price_data[product_id]['history']
hist_dates = [h['date'] for h in history]
hist_prices = [h['price'] for h in history]
fc_dates = [f['date'] for f in result['forecast']]
fc_prices = [f['predicted_price'] for f in result['forecast']]
fc_lower = [f['confidence_interval']['lower'] for f in result['forecast']]
fc_upper = [f['confidence_interval']['upper'] for f in result['forecast']]

plt.figure(figsize=(10, 4))
plt.plot(hist_dates, hist_prices, label='history')
plt.plot(fc_dates, fc_prices, label='forecast')
plt.fill_between(fc_dates, fc_lower, fc_upper, alpha=0.2, label='confidence interval')
plt.xticks(rotation=90, fontsize=6)
plt.legend()
plt.title(f'{product_id} price history + 30-day forecast')
plt.tight_layout()
plt.show()

## 2. Deal Integrity Score

Run it on every sample product to see the range of behavior: a real discount, a markup dressed up as normal, a short-history product, and a deep genuine discount.

In [ ]:
from score_calculator import calculate_deal_score

for pid, p in price_data.items():
    r = calculate_deal_score(p['current_price'], p['history'])
    print(f"{pid}: score={r['score']:>5}  |  {r['reason']}")

## 3. Cross-platform spec matching

Uses the LLM if `OPENAI_API_KEY`/`GEMINI_API_KEY` is set in `../.env`, otherwise falls back to the deterministic heuristic matcher automatically - so this cell works with or without API keys.

In [ ]:
from spec_matcher import match_products

with open(os.path.join(SAMPLE_DIR, 'listings_sample.json')) as f:
    listings = json.load(f)

match_result = match_products(listings['source_product'], listings['candidates'])
for m in match_result['matches']:
    print(m['candidate_id'], '->', m['is_match'], round(m['confidence'], 2), '|', m['notes'])

## 4. Negotiation bot

Drafts a message backed by the Deal Integrity data above.

In [ ]:
from negotiation_logic import draft_negotiation_message

with open(os.path.join(SAMPLE_DIR, 'negotiation_listing_sample.json')) as f:
    listing = json.load(f)['listing']

product = price_data[listing['product_id']]
deal_score_data = calculate_deal_score(product['current_price'], product['history'])

draft = draft_negotiation_message(listing, deal_score_data)
print(draft['message'])
print()
print('proposed price:', draft['proposed_price'])